In [2]:
import re
import joblib
import contractions
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
#final version 
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import contractions
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42


from sklearn.preprocessing import LabelEncoder

#load dataset
df = pd.read_csv("data.csv")
print(df['status'].value_counts()) 


# Encode labels
label_encoder = LabelEncoder()

df["status"] = label_encoder.fit_transform(df["status"])


from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["status"]   # keeps class balance
)

# Custom transformer wrapping clean_data
class TextCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self.clean_data(text) for text in X]

    @staticmethod
    def clean_data(text):
        text = str(text)
        # 0. normalize apostrophes
        text = text.replace("\u2019", "'")
        # 1. lowercase
        text = text.lower()
        # 2. expand contractions
        text = contractions.fix(text)
        # 3. replace names
        text = re.sub(r'@\w+', ' <name> ', text)
        # 4. replace numbers
        text = re.sub(r'\d+', ' <number> ', text)
        # 5. remove punctuation except ?, ! and '
        text = re.sub(r"[^\w\s?!']", "", text)
        return text


pipeline = Pipeline([
    ("cleaner", TextCleaner()),
    ("tfidf", TfidfVectorizer(
        token_pattern=r"\b[a-zA-Z']{2,}\b",
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Train
pipeline.fit(df_train['statement'], df_train["status"])

# Save
joblib.dump(pipeline, "pipeline.joblib")
print("Pipeline saved to pipeline.joblib")

# Load later with:
# pipeline = joblib.load("pipeline.joblib")

status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64
Pipeline saved to pipeline.joblib


In [3]:
joblib.dump(label_encoder, "label_encoder.joblib")  # <-- add this

['label_encoder.joblib']